In [8]:
#!pip install google-genai

In [19]:
from rich.console import Console

In [20]:
from google import genai
import json
from dotenv import load_dotenv
load_dotenv(override=True)

True

In [21]:
def show(text):
    try:
        Console().print(text)
    except Exception:
        print(text)

In [22]:
client = genai.Client(api_key="GEMINI_API_KEY")

In [23]:
# Some lists
todos = []
completed = []

In [24]:
def get_todo_report() -> str:
    result = ""
    for index, todo in enumerate(todos):
        if completed[index]:
            result += f"Todo #{index + 1}: [green][strike]{todo}[/strike][/green]\n"
        else:
            result += f"Todo #{index + 1}: {todo}\n"
    show(result)
    return result
    

In [25]:
get_todo_report()

''

In [26]:
def create_todos(descriptions: list[str]) -> str:
    todos.extend(descriptions)
    completed.extend([False] * len(descriptions))
    return get_todo_report()

In [27]:
def mark_completed(index: int, completion_notes: str) -> str:
    if 1 <= index <= len(todos):
        completed[index - 1] = True
    else:
        return "No todo at this index"
    Console().print(completion_notes)
    return get_todo_report()

In [28]:
todos, completed = [], []

create_todos(["Buy Groceries", "Finish Extra Labs", "Eat Banana"])


Todo #1: Buy Groceries
Todo #2: Finish Extra Labs
Todo #3: Eat Banana

'Todo #1: Buy Groceries\nTodo #2: Finish Extra Labs\nTodo #3: Eat Banana\n'

In [31]:
mark_completed(1, "bought")

bought

Todo #1: Buy Groceries
Todo #2: Finish Extra Labs
Todo #3: Eat Banana

'Todo #1: [green][strike]Buy Groceries[/strike][/green]\nTodo #2: Finish Extra Labs\nTodo #3: Eat Banana\n'

### Function Schemas

In [32]:
create_todos_json = {
    "name": "create_todos",
    "description": "Add new todos from a list of descriptions and return the full list",
    "parameters": {
        "type": "object",
        "properties": {
            "descriptions": {
                'type': 'array',
                'items': {'type': 'string'},
                'title': 'Descriptions'
            }
        },
        "required": ['descriptions'],
        "additionaProperties": False
    }
}

In [33]:
mark_complete_json = {
    "name": "mark_complete",
    "description": "Mark complete the todo at the given position (starting from 1) and return the full list",
    "parameters": {
        'properties': {
            'index': {
                'description': 'The 1-based index of the todo to mark as complete',
                'title': 'Index',
                'type': 'integer'
                },
            'completion_notes': {
                'description': 'Notes about how you completed the todo in rich console markup',
                'title': 'Completion Notes',
                'type': 'string'
                }
            },
        'required': ['index', 'completion_notes'],
        'type': 'object',
        'additionalProperties': False
    }
}

### Registering Tools

In [37]:
tools = [{"type": "function", "function": create_todos_json},
        {"type": "function", "function": mark_complete_json}]

### Executing Tool Calls

In [39]:
def handle_tool_calls(tool_calls):
    results = []
    for tool_call in tool_calls:
        tool_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)
        tool = globals().get(tool_name)
        result = tool(**arguments) if tool else {}
        results.append({"role": "tool", "content": json.dumps(result), "tool_call_id": tool_call.id})

In [55]:
def loop(messages):
    done = False
    while not done:
        response = client.models.generate_content(
            model="gemini-1.5-pro",
            contents=messages,
            config=genai.types.GenerateContentConfig(
                tools=tools
            )
        )

        candidate = response.candidates[0]
        part = candidate.content.parts[0]

        if hasattr(part, "function_call"):
            fn_call = part.function_call
            fn_name = fn_call.name
            fn_args = dict(fn_call.args)

            fn = globals().get(fn_name)
            result = fn(**fn_args) if fn else {}

            messages.append({
                "role": "model",
                "parts": [part]
            })
            messages.append({
                "role": "function",
                "name": fn_name,
                "parts": [{
                    "text": json.dumps(result)
                }]
            })
        else:
            done = True

    print(part.text)


In [56]:
system_message = """
You are given a problem to solve, by using your todo tools to plan a list of steps, then carrying out each step in turn.
Now use the todo list tools, create a plan, carry out the steps, and reply with the solution.
If any quantity isn't provided in the question, then include a step to come up with a reasonable estimate.
Provide your solution in Rich console markup without code blocks.
Do not ask the user questions or clarification; respond only with the answer after using your tools.
"""
user_message = """"
A train leaves Boston at 2:00 pm traveling 60 mph.
Another train leaves New York at 3:00 pm traveling 80 mph toward Boston.
When do they meet?
"""
messages = [{"role": "system", "content": system_message}, {"role": "user", "content": user_message}]

In [57]:
todos, completed = [], []
loop(messages)

ValidationError: 6 validation errors for GenerateContentConfig
tools.0.Tool.type
  Extra inputs are not permitted [type=extra_forbidden, input_value='function', input_type=str]
    For further information visit https://errors.pydantic.dev/2.11/v/extra_forbidden
tools.0.Tool.function
  Extra inputs are not permitted [type=extra_forbidden, input_value={'name': 'create_todos', ...ionaProperties': False}}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.11/v/extra_forbidden
tools.0.callable
  Input should be callable [type=callable_type, input_value={'type': 'function', 'fun...onaProperties': False}}}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.11/v/callable_type
tools.1.Tool.type
  Extra inputs are not permitted [type=extra_forbidden, input_value='function', input_type=str]
    For further information visit https://errors.pydantic.dev/2.11/v/extra_forbidden
tools.1.Tool.function
  Extra inputs are not permitted [type=e